<a href="https://colab.research.google.com/github/alishbakhan21/mmash-sleep-hormones/blob/main/01_load_and_merge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Question: Do saliva melatonin and cortisol differ between bedtime and waking, and do they relate to chronotype?

In [ ]:
!wget -q https://physionet.org/static/published-projects/mmash/multilevel-monitoring-of-activity-and-sleep-in-healthy-people-1.0.0.zip -O mmash.zip

In [ ]:
!ls -lh mmash.zip

-rw-r--r-- 1 root root 23M Mar  9  2024 mmash.zip


In [ ]:
!unzip -q -o mmash.zip -d mmash_raw
!find mmash_raw -name "*.zip" -exec unzip -q -o {} -d mmash \;
!ls mmash

DataPaper


In [ ]:
!ls mmash/*

user_1	 user_12  user_15  user_18  user_20  user_3  user_6  user_9
user_10  user_13  user_16  user_19  user_21  user_4  user_7
user_11  user_14  user_17  user_2   user_22  user_5  user_8


In [12]:
import pandas as pd, glob, os

def load_all(filename):
    frames = []
    for path in sorted(glob.glob(f"mmash/**/{filename}", recursive=True)):
        df = pd.read_csv(path, index_col=0)
        df["user"] = os.path.basename(os.path.dirname(path))
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

In [14]:
saliva = load_all("saliva.csv")
quest  = load_all("questionnaire.csv")
sleep  = load_all("sleep.csv")

for name, df in [("saliva", saliva), ("quest", quest), ("sleep", sleep)]:
    print(name, df.shape, df["user"].nunique(), "users")
    print(df.columns.tolist(), "\n")

saliva (42, 4) 21 users
['SAMPLES', 'Cortisol NORM', 'Melatonin NORM', 'user'] 

quest (22, 20) 22 users
['MEQ', 'STAI1', 'STAI2', 'Pittsburgh', 'Daily_stress', 'BISBAS_bis', 'BISBAS_reward', 'BISBAS_drive', 'BISBAS_fun', 'panas_pos_10', 'panas_pos_14', 'panas_pos_18', 'panas_pos_22', 'panas_pos_9+1', 'panas_neg_10', 'panas_neg_14', 'panas_neg_18', 'panas_neg_22', 'panas_neg_9+1', 'user'] 

sleep (22, 17) 21 users
['In Bed Date', 'In Bed Time', 'Out Bed Date', 'Out Bed Time', 'Onset Date', 'Onset Time', 'Latency', 'Efficiency', 'Total Minutes in Bed', 'Total Sleep Time (TST)', 'Wake After Sleep Onset (WASO)', 'Number of Awakenings', 'Average Awakening Length', 'Movement Index', 'Fragmentation Index', 'Sleep Fragmentation Index', 'user'] 



/tmp/ipykernel_462/3234969424.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True)


In [15]:
all_users = set(quest["user"])
print("No saliva:", all_users - set(saliva["user"]))
print("No sleep:", all_users - set(sleep["user"]))
print(sleep["user"].value_counts().head(3))
print(saliva.isna().sum())

No saliva: {'user_21'}
No sleep: {'user_11'}
user
user_1     2
user_10    1
user_12    1
Name: count, dtype: int64
SAMPLES           0
Cortisol NORM     0
Melatonin NORM    0
user              0
dtype: int64


In [16]:
saliva.head(10)

,SAMPLES,Cortisol NORM,Melatonin NORM,user
0,before sleep,0.034137,1.738385e-08,user_1
1,wake up,0.077939,6.747893e-09,user_1
2,before sleep,0.037046,8.669611e-09,user_10
3,wake up,0.019659,2.573795e-09,user_10
4,before sleep,0.040592,2.038077e-09,user_11
5,wake up,0.015572,9.653476e-09,user_11
6,before sleep,0.155777,3.542804e-09,user_12
7,wake up,0.145431,8.639988e-09,user_12
8,before sleep,0.012278,1.902268e-09,user_13
9,wake up,0.034191,2.295933e-09,user_13


In [17]:
merged = saliva.merge(quest, on="user", how="left")
merged.head()

,SAMPLES,Cortisol NORM,Melatonin NORM,user,MEQ,STAI1,STAI2,Pittsburgh,Daily_stress,BISBAS_bis,...,panas_pos_10,panas_pos_14,panas_pos_18,panas_pos_22,panas_pos_9+1,panas_neg_10,panas_neg_14,panas_neg_18,panas_neg_22,panas_neg_9+1
0,before sleep,0.034137,1.738385e-08,user_1,47.0,41.0,43.0,5.0,23.0,22.0,...,21.0,17.0,12.0,18.0,17.0,11.0,13.0,13.0,10.0,10.0
1,wake up,0.077939,6.747893e-09,user_1,47.0,41.0,43.0,5.0,23.0,22.0,...,21.0,17.0,12.0,18.0,17.0,11.0,13.0,13.0,10.0,10.0
2,before sleep,0.037046,8.669611e-09,user_10,38.0,39.0,46.0,4.0,14.0,19.0,...,28.0,23.0,21.0,18.0,23.0,15.0,23.0,23.0,27.0,12.0
3,wake up,0.019659,2.573795e-09,user_10,38.0,39.0,46.0,4.0,14.0,19.0,...,28.0,23.0,21.0,18.0,23.0,15.0,23.0,23.0,27.0,12.0
4,before sleep,0.040592,2.038077e-09,user_11,38.0,36.0,43.0,7.0,17.0,23.0,...,23.0,16.0,22.0,30.0,13.0,15.0,15.0,11.0,13.0,14.0
